# 04 - Training - Neural Network (PyTorch)

Section **3.4 Training - Neural Network** of the report.

## Brief

Neural network: describe the architecture (layers, activations), optimizer, epochs, and show training curves. Mention the regularization techniques applied (dropout, early stopping, etc.).

- Architecture: layers, activations, dimensions.
- Optimizer, learning rate, batch size, epochs.
- Training and validation curves (loss / metric).
- Regularization: dropout, weight decay, early stopping.
- Device: CPU / CUDA / MPS (Apple Silicon).
- Persist the `state_dict` to `models/` and metrics to `reports/`.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import shap
import torch
import wandb
from dotenv import load_dotenv
from scipy import sparse
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from torch import nn

from diplo_mod_1.constants import CONFIGS, MODELS, PROCESSED, RANDOM_STATE, REPORTS
from diplo_mod_1.schemas.evaluation import evaluate_predictor
from diplo_mod_1.training import (
    NNModelRegistry,
    NNTuner,
    NNTuningConfig,
    TuningHistory,
    WineScoreNet,
    WineScorePredictorNet,
    detect_torch_device,
)

# override=True so re-running this cell after editing .env (without a kernel
# restart) always picks up the new values, instead of keeping whatever was
# already loaded into os.environ earlier in this kernel session.
load_dotenv(override=True)
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Experiment tracking is opt-in: set WANDB_ENABLED=true (in .env or the shell)
# to log this run to Weights & Biases. Off by default so `poe check` / nbmake
# executions don't create a W&B run on every lint/test pass.
WANDB_ENABLED = os.environ.get("WANDB_ENABLED", "false").lower() == "true"

## Step 1 — Load processed dataset (tabular + TF-IDF, concatenated)

Same 44-column tabular export as notebook 03, widened with the TF-IDF matrix (2000 terms, fit on `train`) built for the NN in notebook 02 — the full 2044-column concatenation, not a two-branch architecture. Mirrors XGBoost's already-validated "concatenate and let the model learn" approach (notebook 03 Steps 10-12) rather than re-deriving whether stacking tabular + text helps.

In [ ]:
nn_dir = PROCESSED / "nn"
X_tab = {s: np.load(nn_dir / f"X_tab_{s}.npy") for s in ("train", "val", "test")}
X_txt = {s: sparse.load_npz(nn_dir / f"X_txt_{s}.npz") for s in ("train", "val", "test")}
y = {s: np.load(nn_dir / f"y_{s}.npy") for s in ("train", "val", "test")}

feature_meta = json.loads((nn_dir / "feature_names.json").read_text(encoding="utf-8"))
tabular_feature_names = feature_meta["feature_names"]

tfidf_vectorizer = joblib.load(nn_dir / "tfidf_vectorizer.joblib")
text_feature_names = list(tfidf_vectorizer.get_feature_names_out())
feature_names = tabular_feature_names + text_feature_names

X = {
    s: sparse.hstack([sparse.csr_matrix(X_tab[s]), X_txt[s]], format="csr")
    for s in ("train", "val", "test")
}

print(f"Combined feature count: {len(feature_names)}")
print(f"X_train shape: {X['train'].shape}")

## Step 2 — Baseline model

**Algorithm choice: a feed-forward MLP (`WineScoreNet`).** XGBoost (notebook 03) already showed the enriched tabular+TF-IDF feature set carries most of the available signal (test R² 0.775). The point of this notebook isn't to re-discover that — it's to check whether a differently-shaped model (dense hidden layers with batch norm + dropout, trained by gradient descent instead of boosting) can match or beat it on the same features, for a fair head-to-head in notebook 05. Untuned defaults first (`[128, 64]` hidden sizes) as the number tuning has to beat.

In [ ]:
nn_config_name = os.environ.get("NN_TRAINING_CONFIG", "nn_training.json")
nn_config = NNTuningConfig.from_json(CONFIGS / nn_config_name)
print(f"Loaded NN config: {nn_config_name}")

run_id = f"{Path(nn_config_name).stem}-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"

if WANDB_ENABLED:
    wandb.init(
        project=os.environ.get("WANDB_PROJECT", "diplo-mod-1"),
        name=f"nn-{run_id}",
        group="nn-baseline",
        job_type="hpo",
        config={"nn_config_name": nn_config_name, **nn_config.model_dump()},
    )

baseline = WineScorePredictorNet(
    input_dim=X["train"].shape[1],
    hidden_sizes=[128, 64],
    max_epochs=nn_config.max_epochs,
    early_stopping_patience=nn_config.early_stopping_patience,
    random_state=RANDOM_STATE,
    device=detect_torch_device(),
)
baseline.fit(X["train"], y["train"], X_val=X["val"], y_val=y["val"])

baseline_pred = baseline.predict(X["val"])
print(f"Baseline val RMSE: {root_mean_squared_error(y['val'], baseline_pred):.4f}")
print(f"Baseline val MAE:  {mean_absolute_error(y['val'], baseline_pred):.4f}")
print(f"Baseline val R2:   {r2_score(y['val'], baseline_pred):.4f}")

## Step 3 — Hyperparameter tuning (Optuna)

Bayesian search (TPE sampler) via `NNTuner` (`src/diplo_mod_1/training/`), configured from a JSON file in `configs/` — default `nn_training.json`, override with the `NN_TRAINING_CONFIG` env var, mirroring `XGBOOST_TUNING_CONFIG` from notebook 03. Search space: `architecture` (hidden-layer sizes as a categorical choice among `"128_64"`, `"256_64"`, `"512_128_32"` — Optuna doesn't tune variable-depth networks directly), `dropout`, `learning_rate`, `weight_decay`, `batch_size`. Each trial fits with early stopping against `val` (same split doubling as objective + early-stopping monitor as XGBoost's Step 3) — `test` stays untouched until Step 5.

The default budget (`n_trials=10`, `max_epochs=30`, `early_stopping_patience=5`) is deliberately small: a full gradient-descent training loop per trial is far more expensive than one boosted-tree fit, so a 50-trial XGBoost-sized search here would be prohibitively slow for a first pass. Widen `configs/nn_training.json` (or point `NN_TRAINING_CONFIG` at a new file) once this budget's result establishes a baseline worth spending more search time on.

In [ ]:
def log_trial_to_wandb(study: optuna.Study, trial: optuna.trial.FrozenTrial) -> None:
    wandb.log({"trial": trial.number, "val_rmse": trial.value, **trial.params})


tuner = NNTuner(nn_config)
callbacks = [log_trial_to_wandb] if WANDB_ENABLED else None
study = tuner.tune(X["train"], y["train"], X["val"], y["val"], callbacks=callbacks)

print(f"Best val RMSE: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

if WANDB_ENABLED:
    wandb.log(
        {
            "best_val_rmse": study.best_value,
            **{f"best_{k}": v for k, v in study.best_params.items()},
        }
    )

## Step 4 — Final model + per-epoch W&B logging

Refit on `study.best_params`, same early-stopping setup as the baseline — picks the epoch that generalizes best rather than a fixed epoch count. Unlike Step 3 (which only logs one RMSE per trial), this refit logs every epoch's train/val loss, so the winning configuration's full training curve is visible, not just its terminal value.

In [ ]:
def log_epoch_to_wandb(epoch: int, train_loss: float, val_loss: float) -> None:
    wandb.log({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss})


epoch_callbacks = [log_epoch_to_wandb] if WANDB_ENABLED else None
best_model = tuner._make_model(X["train"].shape[1], study.best_params)
best_model.fit(X["train"], y["train"], X_val=X["val"], y_val=y["val"], callbacks=epoch_callbacks)
print(f"Best epoch: {best_model.best_epoch_} / {len(best_model.train_losses_)} trained")

## Step 5 — Evaluate on train / val / test

In [ ]:
splits = {
    "train": (X["train"], y["train"]),
    "val": (X["val"], y["val"]),
    "test": (X["test"], y["test"]),
}
result = evaluate_predictor(best_model, splits, model_type="neural_net")

for m in result.metrics:
    print(f"{m.split:5s}  RMSE={m.rmse:.4f}  MAE={m.mae:.4f}  R2={m.r2:.4f}")
    if WANDB_ENABLED:
        wandb.log({f"{m.split}_rmse": m.rmse, f"{m.split}_mae": m.mae, f"{m.split}_r2": m.r2})

## Step 6 — Training curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(best_model.train_losses_, label="train")
ax.plot(best_model.val_losses_, label="val")
ax.axvline(best_model.best_epoch_, color="crimson", lw=1.5, linestyle="--", label="best epoch")
ax.set_xlabel("epoch")
ax.set_ylabel("MSE loss")
ax.set_title("Training curves")
ax.legend()
plt.tight_layout()
plt.show()

if WANDB_ENABLED:
    wandb.log({"training_curves": wandb.Image(fig)})

## Step 7 — Residual analysis

Residuals (`y_test - predicted`) for `best_model` on `test`: same 3-subplot pattern as notebook 03 Step 7 (residuals vs. predicted, residual distribution, residuals vs. actual) — checks for heteroscedasticity, skew, and systematic over/under-prediction at specific `points` ranges. Run before Step 8 persists/closes the W&B run, so the plot gets logged too when `WANDB_ENABLED=true`.

In [ ]:
test_pred = best_model.predict(X["test"])
residuals = y["test"] - test_pred

print(f"Residual mean:   {residuals.mean():.4f}  (0 = unbiased)")
print(f"Residual std:    {residuals.std():.4f}")
print(f"Residual |max|:  {np.abs(residuals).max():.4f}")

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
ax.scatter(test_pred, residuals, alpha=0.15, s=8, color="steelblue")
ax.axhline(0, color="crimson", lw=1.5, linestyle="--")
ax.set_xlabel("predicted points")
ax.set_ylabel("residual (actual - predicted)")
ax.set_title("Residuals vs. predicted")

ax = axes[1]
ax.hist(residuals, bins=50, color="steelblue")
ax.axvline(0, color="crimson", lw=1.5, linestyle="--")
ax.set_xlabel("residual")
ax.set_ylabel("count")
ax.set_title("Residual distribution")

ax = axes[2]
ax.scatter(y["test"], residuals, alpha=0.15, s=8, color="steelblue")
ax.axhline(0, color="crimson", lw=1.5, linestyle="--")
ax.set_xlabel("actual points")
ax.set_ylabel("residual (actual - predicted)")
ax.set_title("Residuals vs. actual")

plt.tight_layout()
plt.show()

if WANDB_ENABLED:
    wandb.log({"residual_analysis": wandb.Image(fig)})

## Step 8 — Persist model and metrics

`NNModelRegistry` (`src/diplo_mod_1/training/nn_registry.py`) mirrors `ModelRegistry`'s contract without sharing a base class — `state_dict` + architecture config vs. a picklable XGBoost object are different enough serialization mechanics that a shared abstraction for exactly these two call sites would be premature. Each run's checkpoint is saved as `models/<run_id>.pt`, and `models/nn_best.pt` always points at whichever run has the lowest **test**-split RMSE on record. `reports/nn_metrics.json` holds the full run history, consumed by notebook 05 for the model comparison.

In [ ]:
run_record, history = NNModelRegistry.save_run(
    MODELS,
    REPORTS / "nn_metrics.json",
    best_model,
    run_id,
    nn_config_name,
    study.best_params,
    result,
)
print(f"Model saved to {MODELS / run_record.model_filename}")
print(f"Best run so far: {history.best_run_id} -> models/nn_best.pt")

if WANDB_ENABLED:
    artifact = wandb.Artifact("nn_model", type="model")
    artifact.add_file(str(MODELS / run_record.model_filename))
    wandb.log_artifact(artifact)
    wandb.finish()

## Step 9 — Compare all NN runs

Every run recorded in `reports/nn_metrics.json` (across all `configs/*.json` search spaces tried), side by side. The winning row matches `models/nn_best.pt`.

In [ ]:
history_path = REPORTS / "nn_metrics.json"
all_runs = TuningHistory.model_validate_json(history_path.read_text(encoding="utf-8"))

comparison = pd.DataFrame(
    [
        {
            "run_id": r.run_id,
            "tuning_config": r.tuning_config,
            "train_r2": next(m.r2 for m in r.metrics if m.split == "train"),
            "train_mae": next(m.mae for m in r.metrics if m.split == "train"),
            "train_rmse": next(m.rmse for m in r.metrics if m.split == "train"),
            "test_r2": next(m.r2 for m in r.metrics if m.split == "test"),
            "test_rmse": next(m.rmse for m in r.metrics if m.split == "test"),
            "test_mae": next(m.mae for m in r.metrics if m.split == "test"),
            "val_r2": next(m.r2 for m in r.metrics if m.split == "val"),
            "val_rmse": next(m.rmse for m in r.metrics if m.split == "val"),
            "val_mae": next(m.mae for m in r.metrics if m.split == "val"),
            "is_best": r.run_id == all_runs.best_run_id,
        }
        for r in all_runs.runs
    ]
).sort_values("test_r2", ascending=False)

comparison

## Step 10 — SHAP explainability for the NN

Loads `models/nn_best.pt` **fresh from disk** (not whichever model happens to be in kernel memory) and reconstructs `WineScoreNet` from the saved architecture config, same discipline as notebook 03 Step 13. `shap.GradientExplainer` runs against a background + sample drawn from `test`, densifying only those rows (not the whole sparse test matrix) — same pattern `SparseTabularDataset` uses to avoid materializing the ~665MB dense TF-IDF block. Own W&B run (`group="nn-shap"`, `job_type="explainability"`).

A standalone probe against a real `WineScoreNet` (same discipline that caught XGBoost's `base_score` shap-compatibility bug outside the notebook first, see notebook 03's `shap_support.py`) found that `shap.GradientExplainer` raises `IndexError: too many indices for tensor of dimension 1` directly against `WineScoreNet` — it indexes model outputs as `outputs[:, i]`, but `WineScoreNet.forward` squeezes its single regression output to a flat `(batch,)` tensor. The fix is a trivial wrapper that re-adds the trailing size-1 output dimension, scoped to this cell rather than changed in `nn_model.py` — SHAP is the only caller that needs it; `fit`/`predict`'s flat-array contract is untouched.

In [ ]:
SHAP_SAMPLE_SIZE = 500
SHAP_BACKGROUND_SIZE = 100


class _GradientExplainerOutput(nn.Module):
    """Adds back the output dim shap.GradientExplainer needs.

    WineScoreNet.forward returns a flat ``(batch,)`` tensor (squeezed for
    fit/predict); GradientExplainer indexes outputs as ``outputs[:, i]``
    and raises IndexError against a 1-D tensor. Confirmed via a standalone
    probe before wiring this in — see the markdown cell above.
    """

    def __init__(self, model: nn.Module) -> None:
        super().__init__()
        self.model = model

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.model(x).unsqueeze(-1)


checkpoint = torch.load(MODELS / "nn_best.pt", map_location="cpu", weights_only=False)
best_overall_nn = WineScoreNet(
    input_dim=checkpoint["input_dim"],
    hidden_sizes=checkpoint["hidden_sizes"],
    dropout=checkpoint["dropout"],
)
best_overall_nn.load_state_dict(checkpoint["state_dict"])
best_overall_nn.eval()

rng = np.random.default_rng(RANDOM_STATE)
n_test = X["test"].shape[0]
background_idx = rng.choice(n_test, size=SHAP_BACKGROUND_SIZE, replace=False)
sample_idx = rng.choice(n_test, size=min(SHAP_SAMPLE_SIZE, n_test), replace=False)

background_rows = X["test"][background_idx]
sample_rows = X["test"][sample_idx]
if sparse.issparse(background_rows):
    background_rows = background_rows.toarray()
    sample_rows = sample_rows.toarray()

background = torch.from_numpy(np.asarray(background_rows, dtype=np.float32))
X_sample_tensor = torch.from_numpy(np.asarray(sample_rows, dtype=np.float32))

explainer = shap.GradientExplainer(_GradientExplainerOutput(best_overall_nn), background)
shap_values = explainer.shap_values(X_sample_tensor)
if isinstance(shap_values, list):
    shap_values = shap_values[0]
shap_values = np.asarray(shap_values).reshape(len(sample_idx), -1)

mean_abs_shap = np.abs(shap_values).mean(axis=0)
top_idx = np.argsort(mean_abs_shap)[::-1][:30]
shap_table = pd.DataFrame(
    {"feature": [feature_names[i] for i in top_idx], "mean_abs_shap": mean_abs_shap[top_idx]}
)
print(shap_table.head(15).to_string(index=False))

shap.summary_plot(
    shap_values, X_sample_tensor.numpy(), feature_names=feature_names, max_display=25, show=False
)
beeswarm_fig = plt.gcf()
plt.tight_layout()
plt.show()

if WANDB_ENABLED:
    wandb.init(
        project=os.environ.get("WANDB_PROJECT", "diplo-mod-1"),
        name=f"shap-nn-{history.best_run_id}",
        group="nn-shap",
        job_type="explainability",
        config={
            "explained_run_id": history.best_run_id,
            "sample_size": min(SHAP_SAMPLE_SIZE, n_test),
        },
    )
    wandb.log(
        {
            "shap_summary_beeswarm": wandb.Image(beeswarm_fig),
            "shap_top_features": wandb.Table(dataframe=shap_table),
        }
    )
    wandb.finish()

## Step 11 — Design notes (report)

**Algorithm choice:** a feed-forward MLP (`WineScoreNet` — `[Linear -> BatchNorm1d -> ReLU -> Dropout]` per hidden layer, then `Linear(*, 1)`). Not chosen to out-perform XGBoost on priors — notebook 03 already found gradient-boosted trees strong on this feature set — but to give the report a genuinely different modeling approach (dense representation learning by gradient descent, not axis-aligned tree splits) over the identical 2044-column tabular+TF-IDF matrix, so notebook 05's comparison is apples-to-apples on features and differs only in model family.

**Feature set:** deliberately identical to XGBoost's winning tabular+TF-IDF input (notebook 03 Steps 10-12) — 44 tabular columns concatenated with the 2000-term TF-IDF matrix from notebook 02, via `scipy.sparse.hstack`. `SparseTabularDataset` densifies one row at a time inside `DataLoader.__getitem__` rather than the whole matrix up front, since the TF-IDF block alone is ~665MB dense at ~1% non-zero.

**Architecture and tuning process:** `NNTuner` (`src/diplo_mod_1/training/nn_tuner.py`) mirrors `XGBoostTuner`'s shape — Optuna TPE search, configured from `configs/nn_training.json` (override via `NN_TRAINING_CONFIG`), early-stopped against `val`. The search space covers hidden-layer depth/width as a categorical `architecture` choice (`"128_64"`, `"256_64"`, `"512_128_32"`) plus `dropout`, `learning_rate`, `weight_decay`, `batch_size`. The default budget (`n_trials=10`, `max_epochs=30`, `early_stopping_patience=5`) is intentionally smaller than XGBoost's (`n_trials=50`): one NN trial is a full gradient-descent training loop, an order of magnitude slower per trial than a boosted-tree fit, so a first-pass budget needs to stay small enough to actually finish before deciding whether it's worth widening.

**Validation strategy:** same holdout split as notebook 03 (stratified `train`/`val`/`test` from notebook 02), not k-fold — `val` drives both the Optuna objective and early stopping, `test` is touched only for final reported metrics via `evaluate_predictor` (shared with notebook 03 through the `WineScorePredictor` protocol), keeping it a fair basis for notebook 05's head-to-head.

**Device:** `detect_torch_device()` (`src/diplo_mod_1/training/device.py`) checks `torch.cuda.is_available()` then `torch.backends.mps.is_available()` then falls back to CPU — trustworthy here (unlike XGBoost's separate CUDA probe) because torch's own device APIs aren't second-guessing a differently-bundled runtime; this is the only place in the project MPS is a real target, since XGBoost has no Apple Silicon GPU backend.

**Persistence:** every run's checkpoint is versioned (`models/<run_id>.pt`, via `NNModelRegistry`), with `models/nn_best.pt` always pointing at the lowest-test-RMSE run recorded so far; `reports/nn_metrics.json` accumulates the full run history the same way `xgboost_metrics.json` does, so different search-space experiments stay comparable (Step 9). Optionally logged to Weights & Biases end-to-end (per-trial in Step 3, per-epoch for the final refit in Step 4, plus the training-curve and residual-analysis plots and the model artifact) when `WANDB_ENABLED=true`.

**Explainability (Step 10):** `shap.GradientExplainer` against `models/nn_best.pt` loaded fresh from disk, on a background/sample drawn from `test`. Required one fix found by probing `GradientExplainer` against a real `WineScoreNet` before wiring it into the notebook (same discipline as the XGBoost `base_score` shim in notebook 03): `WineScoreNet.forward` squeezes its output to a flat `(batch,)` tensor for `fit`/`predict`, but `GradientExplainer` indexes outputs as `outputs[:, i]` and raises `IndexError` against a 1-D tensor. Fixed with a one-off `nn.Module` wrapper local to the SHAP cell (adds back the trailing size-1 output dim) rather than changing `nn_model.py` — SHAP is the only caller that needs it.

**Results — pending the first full run.** This notebook was built and verified via `poe lint`/`poe typecheck` plus a standalone data-loading smoke check and the `GradientExplainer` probe above, but was not executed end-to-end (training runs, including any live W&B logging, are the user's to trigger). Once run, this section should be updated with: baseline vs. tuned val/test RMSE/MAE/R², the winning architecture and hyperparameters, the top SHAP features by mean |SHAP| for the NN, and a comparison against XGBoost's SHAP results (notebook 03 Step 13 found TF-IDF terms — "gorgeous", "lacks", "dominates", "generic" — dominating over tabular columns; worth checking whether the NN's dense representation surfaces the same terms or spreads importance differently across the 2044 features).

**Known open question:** same as notebook 03 — `NNModelRegistry`'s "lowest test RMSE wins" selection doesn't account for overfit gap or statistical significance between close results, relevant if a future architecture/tuning experiment lands within noise of the current best.